# Demo 06: LoRA fine-tuning for IT-support ticket classification

Train a BF16 LoRA adapter for the structured JSON ticket-classification task. This phase uses only training and validation records; the held-out test split is intentionally not loaded.

The next cell imports every library used in the notebook and defines reusable validation, generation, reporting, and callback helpers.

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import numpy as np
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback, set_seed
from trl import SFTConfig, SFTTrainer


def find_project_root(start_directory: Path) -> Path:
    """Return the repository root containing the project instructions."""
    for candidate in (start_directory, *start_directory.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'requirements.txt').is_file():
            return candidate
    raise RuntimeError('Could not find the project root. Start JupyterLab from the repository root.')


def select_device(requested_device: str) -> torch.device:
    """Select MPS when available or explicitly select CPU."""
    if requested_device not in {'auto', 'mps', 'cpu'}:
        raise ValueError("DEVICE_REQUEST must be 'auto', 'mps', or 'cpu'.")
    if requested_device == 'cpu':
        return torch.device('cpu')
    if requested_device == 'mps':
        if not torch.backends.mps.is_built() or not torch.backends.mps.is_available():
            raise RuntimeError('MPS was requested but is unavailable in this runtime.')
        return torch.device('mps')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    print('Warning: MPS is unavailable. Falling back to CPU; training will be slower.')
    return torch.device('cpu')


def require_bfloat16_mps_support(device: torch.device) -> None:
    """Raise an actionable error when BF16 fine-tuning cannot use MPS."""
    if device.type != 'mps':
        raise RuntimeError("BF16 fine-tuning requires an available MPS device. Set DEVICE_REQUEST to 'mps' on a supported Apple Silicon runtime.")
    macos_version = platform.mac_ver()[0]
    macos_major = int(macos_version.split('.')[0]) if macos_version else 0
    if macos_major < 14:
        raise RuntimeError(f"BF16 fine-tuning on MPS requires macOS 14 or later; detected {macos_version or 'an unknown version'}.")


def to_prompt_completion(record: dict) -> dict:
    """Convert a ticket record to TRL prompt/completion format."""
    return {'prompt': [record['messages'][0]], 'completion': [record['messages'][1]]}


def validate_split(split_name: str, dataset) -> set[str]:
    """Validate ticket records and return normalized ticket texts without printing them."""
    ticket_texts = set()
    for index, record in enumerate(dataset):
        messages = record.get('messages')
        if not isinstance(messages, list) or len(messages) != 2:
            raise ValueError(f'{split_name} record {index} must contain exactly two messages.')
        if [message.get('role') for message in messages] != ['user', 'assistant']:
            raise ValueError(f'{split_name} record {index} must have user and assistant roles.')
        ticket_text = messages[0].get('content', '').strip()
        if not ticket_text:
            raise ValueError(f'{split_name} record {index} has empty ticket text.')
        if ticket_text in ticket_texts:
            raise ValueError(f'{split_name} contains duplicate ticket text at record {index}.')
        ticket_texts.add(ticket_text)
        try:
            output = json.loads(messages[1].get('content', ''))
        except json.JSONDecodeError as error:
            raise ValueError(f'{split_name} record {index} has invalid assistant JSON.') from error
        if not isinstance(output, dict) or set(output) != {'category', 'severity', 'summary'}:
            raise ValueError(f'{split_name} record {index} must contain exactly category, severity, and summary.')
        if output['category'] not in ALLOWED_CATEGORIES or output['severity'] not in ALLOWED_SEVERITIES:
            raise ValueError(f'{split_name} record {index} has an unsupported category or severity.')
        if not isinstance(output['summary'], str) or not output['summary'].strip():
            raise ValueError(f'{split_name} record {index} has an empty summary.')
    return ticket_texts


def calculate_validation_generation_metrics(model, tokenizer, validation_dataset) -> dict[str, float]:
    """Generate validation responses and calculate task-specific aggregate metrics."""
    valid_json_count = category_correct_count = severity_correct_count = joint_correct_count = 0
    model.eval()
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    try:
        for start_index in range(0, len(validation_dataset), VALIDATION_GENERATION_BATCH_SIZE):
            batch_records = validation_dataset.select(range(start_index, min(start_index + VALIDATION_GENERATION_BATCH_SIZE, len(validation_dataset))))
            prompt_texts = [tokenizer.apply_chat_template([record['messages'][0]], tokenize=False, add_generation_prompt=True, enable_thinking=False) for record in batch_records]
            model_inputs = {name: tensor.to(DEVICE) for name, tensor in tokenizer(prompt_texts, return_tensors='pt', padding=True).items()}
            with torch.inference_mode():
                output_ids = model.generate(**model_inputs, max_new_tokens=GENERATION_MAX_NEW_TOKENS, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
            continuation_start = model_inputs['input_ids'].shape[-1]
            predictions = tokenizer.batch_decode(output_ids[:, continuation_start:], skip_special_tokens=True)
            for record, prediction in zip(batch_records, predictions):
                try:
                    generated_output = json.loads(prediction.strip())
                    valid_json = isinstance(generated_output, dict)
                except json.JSONDecodeError:
                    generated_output = {}
                    valid_json = False
                reference_output = json.loads(record['messages'][1]['content'])
                category_correct = valid_json and generated_output.get('category') == reference_output['category']
                severity_correct = valid_json and generated_output.get('severity') == reference_output['severity']
                valid_json_count += valid_json
                category_correct_count += category_correct
                severity_correct_count += severity_correct
                joint_correct_count += category_correct and severity_correct
    finally:
        tokenizer.padding_side = original_padding_side
    record_count = len(validation_dataset)
    return {'category_accuracy': category_correct_count / record_count, 'severity_accuracy': severity_correct_count / record_count, 'joint_accuracy': joint_correct_count / record_count, 'valid_json_rate': valid_json_count / record_count}


def display_epoch_metrics(metrics: dict[str, float]) -> None:
    """Display the task-specific validation metrics immediately after one epoch."""
    table_lines = [
        '| Epoch | Training Loss | Validation Loss | Category Accuracy | Severity Accuracy | Category+Severity Accuracy | Valid JSON Rate |',
        '| ---: | ---: | ---: | ---: | ---: | ---: | ---: |',
        f"| {metrics['epoch']:.0f} | {metrics['training_loss']:.6f} | {metrics['validation_loss']:.6f} | {metrics['category_accuracy']:.1%} | {metrics['severity_accuracy']:.1%} | {metrics['joint_accuracy']:.1%} | {metrics['valid_json_rate']:.1%} |",
    ]
    display(Markdown('\n'.join(table_lines)))


class ValidationGenerationMetricsCallback(TrainerCallback):
    """Record deterministic ticket-classification metrics after each validation epoch."""
    def __init__(self, model, tokenizer, validation_dataset):
        self.model = model
        self.tokenizer = tokenizer
        self.validation_dataset = validation_dataset
        self.epoch_metrics = []

    def on_evaluate(self, args, state, control, **kwargs):
        evaluation_metrics = kwargs.get('metrics', {})
        generation_metrics = calculate_validation_generation_metrics(self.model, self.tokenizer, self.validation_dataset)
        training_loss = next((entry['loss'] for entry in reversed(state.log_history) if 'loss' in entry), float('nan'))
        epoch_metrics = {'epoch': state.epoch, 'training_loss': training_loss, 'validation_loss': evaluation_metrics['eval_loss'], **generation_metrics}
        evaluation_metrics['eval_joint_accuracy'] = generation_metrics['joint_accuracy']
        self.epoch_metrics.append(epoch_metrics)
        display_epoch_metrics(epoch_metrics)
        return control


## Local artifacts

The next cell resolves the project paths, selects the version 2 training and validation files, and verifies that all required local model artifacts are complete before loading anything.

In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd().resolve())
MODEL_DIRECTORY = PROJECT_ROOT / 'artifacts' / 'models' / 'Qwen3-1.7B'
DATASET_DIRECTORY = PROJECT_ROOT / 'ticket-classification' / 'artifacts' / 'datasets'
# Use dataset version 2 for this experiment; version 1 files remain available for comparison.
TRAIN_FILE = DATASET_DIRECTORY / 'train_dataset_v2.jsonl'
VALIDATION_FILE = DATASET_DIRECTORY / 'validation_dataset_v2.jsonl'

for required_path in (MODEL_DIRECTORY / 'config.json', MODEL_DIRECTORY / 'tokenizer.json', TRAIN_FILE, VALIDATION_FILE):
    if not required_path.is_file():
        raise FileNotFoundError(f'Missing required local artifact: {required_path}')
single_weight_file = MODEL_DIRECTORY / 'model.safetensors'
weight_index_file = MODEL_DIRECTORY / 'model.safetensors.index.json'
if not single_weight_file.is_file() and not weight_index_file.is_file():
    raise FileNotFoundError(f'Missing model weights in {MODEL_DIRECTORY}. Expected model.safetensors or model.safetensors.index.json.')
if weight_index_file.is_file():
    shard_names = set(json.loads(weight_index_file.read_text(encoding='utf-8'))['weight_map'].values())
    missing_shards = sorted(name for name in shard_names if not (MODEL_DIRECTORY / name).is_file())
    if missing_shards:
        raise FileNotFoundError(f'Missing model weight shards in {MODEL_DIRECTORY}: {missing_shards}')
print(f'Model directory: {MODEL_DIRECTORY}')
print(f'Dataset directory: {DATASET_DIRECTORY}')


## Configuration and execution environment

The next cell exposes the reproducible training and evaluation configuration, selects MPS, and checks the BF16 macOS requirements. The checkpoint policy is explicit: the best adapter is the epoch with the highest validation Category+Severity Accuracy.

In [ ]:
SEED = 42
MODEL_DTYPE = torch.bfloat16
DEVICE_REQUEST = 'auto'  # Allowed values: 'auto', 'mps', 'cpu'

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

LEARNING_RATE = 1e-4
NUM_TRAIN_EPOCHS = 8
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_SEQUENCE_LENGTH = 512
GENERATION_MAX_NEW_TOKENS = 128
VALIDATION_GENERATION_BATCH_SIZE = 4

TRAINING_OUTPUT_DIRECTORY = PROJECT_ROOT / 'ticket-classification' / 'artifacts' / 'models' / 'demo06-qwen3-1.7b-ticket-classification-lora'
ADAPTER_OUTPUT_DIRECTORY = TRAINING_OUTPUT_DIRECTORY / 'adapter'
ALLOWED_CATEGORIES = {'APP-01', 'AUTH-01', 'DB-01', 'DB-02', 'DEV-01', 'MAIL-01', 'MOB-01', 'NET-01', 'SEC-01', 'SRV-01'}
ALLOWED_SEVERITIES = {'P1', 'P2', 'P3', 'P4'}

DEVICE = select_device(DEVICE_REQUEST)
require_bfloat16_mps_support(DEVICE)
set_seed(SEED)
print(f'PyTorch version: {torch.__version__}')
print(f'MPS built: {torch.backends.mps.is_built()}')
print(f'MPS available: {torch.backends.mps.is_available()}')
print(f'Selected device: {DEVICE}')
print(f'Model dtype: {MODEL_DTYPE}')


## Load and validate datasets

The next cell loads only the training and validation splits, validates their structured JSON contract and separation, and converts them to TRL prompt/completion records.

In [ ]:
raw_datasets = load_dataset('json', data_files={'train': str(TRAIN_FILE), 'validation': str(VALIDATION_FILE)})
train_ticket_texts = validate_split('train', raw_datasets['train'])
validation_ticket_texts = validate_split('validation', raw_datasets['validation'])
if train_ticket_texts & validation_ticket_texts:
    raise ValueError('Training and validation splits share exact ticket text.')
sft_datasets = raw_datasets.map(to_prompt_completion, remove_columns=raw_datasets['train'].column_names)
print(f"Training records: {len(sft_datasets['train'])}")
print(f"Validation records: {len(sft_datasets['validation'])}")
print('The held-out test split is intentionally not loaded by this notebook.')


## Train and save the best adapter

The next cell loads Qwen3-1.7B in BF16, adds the LoRA adapter, trains with the existing settings, and evaluates validation loss plus task metrics after every epoch. Checkpoint selection maximizes validation Category+Severity Accuracy; the restored best adapter and tokenizer are then saved locally.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIRECTORY, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_DIRECTORY, dtype=MODEL_DTYPE, local_files_only=True).to(DEVICE)
model.config.use_cache = False
if model.config.model_type != 'qwen3':
    raise RuntimeError(f'Expected a Qwen3 base model, got {model.config.model_type!r}.')
lora_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias='none', target_modules=LORA_TARGET_MODULES)
model = get_peft_model(model, lora_config, autocast_adapter_dtype=False)
training_arguments = SFTConfig(output_dir=str(TRAINING_OUTPUT_DIRECTORY), learning_rate=LEARNING_RATE, num_train_epochs=NUM_TRAIN_EPOCHS, per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE, per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, max_length=MAX_SEQUENCE_LENGTH, eval_strategy='epoch', save_strategy='epoch', logging_strategy='steps', logging_steps=5, save_total_limit=1, load_best_model_at_end=True, metric_for_best_model='eval_joint_accuracy', greater_is_better=True, completion_only_loss=True, gradient_checkpointing=True, bf16=True, bf16_full_eval=True, dataloader_pin_memory=False, optim='adamw_torch', report_to='none', seed=SEED)
validation_metrics_callback = ValidationGenerationMetricsCallback(model, tokenizer, raw_datasets['validation'])
trainer = SFTTrainer(model=model, args=training_arguments, train_dataset=sft_datasets['train'], eval_dataset=sft_datasets['validation'], processing_class=tokenizer, callbacks=[validation_metrics_callback])
floating_dtypes = {parameter.dtype for parameter in trainer.model.parameters() if parameter.is_floating_point()}
trainable_dtypes = {parameter.dtype for parameter in trainer.model.parameters() if parameter.requires_grad}
if floating_dtypes != {MODEL_DTYPE} or trainable_dtypes != {MODEL_DTYPE}:
    raise RuntimeError(f'Expected BF16 floating and trainable parameters, got {floating_dtypes} and {trainable_dtypes}.')
trainer.model.print_trainable_parameters()
print(f'Trainable LoRA parameter dtypes: {trainable_dtypes}')
train_result = trainer.train()
trainer.save_model(ADAPTER_OUTPUT_DIRECTORY)
tokenizer.save_pretrained(ADAPTER_OUTPUT_DIRECTORY)
print(f'Training loss: {train_result.training_loss:.4f}')
print(f'Adapter saved to: {ADAPTER_OUTPUT_DIRECTORY}')


## Review losses and validation metrics

The next cell plots training and validation loss, then renders the complete per-epoch task-metric table. These results do not change checkpoint selection after training has completed.

In [ ]:
training_logs = [entry for entry in trainer.state.log_history if 'loss' in entry and 'epoch' in entry]
validation_logs = [entry for entry in trainer.state.log_history if 'eval_loss' in entry and 'epoch' in entry]
if not training_logs or not validation_logs:
    raise RuntimeError('Training or validation loss logs are unavailable.')
fig, axis = plt.subplots(figsize=(10, 6))
axis.plot([entry['epoch'] for entry in training_logs], [entry['loss'] for entry in training_logs], color='#2563EB', linewidth=2, marker='o', markersize=4, label='Training loss')
axis.plot([entry['epoch'] for entry in validation_logs], [entry['eval_loss'] for entry in validation_logs], color='#DC2626', linewidth=2, marker='s', markersize=6, label='Validation loss')
axis.set_title('Training and Validation Loss by Epoch', fontsize=14, fontweight='bold')
axis.set_xlabel('Epoch')
axis.set_ylabel('Loss')
axis.grid(True, linestyle='--', linewidth=0.7, alpha=0.55)
axis.legend(frameon=True)
axis.set_xticks([entry['epoch'] for entry in validation_logs])
plt.tight_layout()
plt.show()

if len(validation_metrics_callback.epoch_metrics) != len(validation_logs):
    raise RuntimeError('Task-specific validation metrics are unavailable for one or more epochs.')
table_lines = [
    '| Epoch | Training Loss | Validation Loss | Category Accuracy | Severity Accuracy | Category+Severity Accuracy | Valid JSON Rate |',
    '| ---: | ---: | ---: | ---: | ---: | ---: | ---: |',
]
for metrics in validation_metrics_callback.epoch_metrics:
    table_lines.append(f"| {metrics['epoch']:.0f} | {metrics['training_loss']:.6f} | {metrics['validation_loss']:.6f} | {metrics['category_accuracy']:.1%} | {metrics['severity_accuracy']:.1%} | {metrics['joint_accuracy']:.1%} | {metrics['valid_json_rate']:.1%} |")
display(Markdown('\n'.join(table_lines)))


## Inspect severity errors

The next cell regenerates validation responses from the restored best adapter and visualizes the P1–P4 severity confusion matrix. It is an analysis-only step and does not change the trained model.

In [ ]:
SEVERITY_LABELS = ["P1", "P2", "P3", "P4"]
severity_to_index = {severity: index for index, severity in enumerate(SEVERITY_LABELS)}

confusion_matrix = np.zeros((len(SEVERITY_LABELS), len(SEVERITY_LABELS)), dtype=int)
invalid_or_unknown_predictions = 0

trainer.model.eval()

for record in raw_datasets["validation"]:
    prompt_text = tokenizer.apply_chat_template(
        [record["messages"][0]],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    model_inputs = {
        name: tensor.to(DEVICE)
        for name, tensor in tokenizer(prompt_text, return_tensors="pt").items()
    }

    with torch.inference_mode():
        output_ids = trainer.model.generate(
            **model_inputs,
            max_new_tokens=GENERATION_MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    prediction = tokenizer.decode(
        output_ids[0, model_inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()

    reference = json.loads(record["messages"][1]["content"])
    true_severity = reference["severity"]

    try:
        predicted_output = json.loads(prediction)
        predicted_severity = predicted_output.get("severity")
    except json.JSONDecodeError:
        predicted_severity = None

    if predicted_severity not in severity_to_index:
        invalid_or_unknown_predictions += 1
        continue

    confusion_matrix[
        severity_to_index[true_severity],
        severity_to_index[predicted_severity],
    ] += 1

fig, axis = plt.subplots(figsize=(7, 6))
image = axis.imshow(confusion_matrix, cmap="Blues")

axis.set(
    xticks=np.arange(len(SEVERITY_LABELS)),
    yticks=np.arange(len(SEVERITY_LABELS)),
    xticklabels=SEVERITY_LABELS,
    yticklabels=SEVERITY_LABELS,
    xlabel="Predicted severity",
    ylabel="True severity",
    title="Validation Severity Confusion Matrix",
)

for row_index in range(len(SEVERITY_LABELS)):
    for column_index in range(len(SEVERITY_LABELS)):
        value = confusion_matrix[row_index, column_index]
        axis.text(
            column_index,
            row_index,
            str(value),
            ha="center",
            va="center",
            color="white" if value > confusion_matrix.max() / 2 else "black",
        )

fig.colorbar(image, ax=axis, label="Ticket count")
plt.tight_layout()
plt.show()

print(f"Invalid or unknown severity predictions: {invalid_or_unknown_predictions}")
print(f"Severity accuracy: {np.trace(confusion_matrix) / len(raw_datasets['validation']):.1%}")